In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import re
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [ ]:
df = pd.read_csv("../data/papers.csv")
total_papers = len(df)
df.columns

In [ ]:
age_bias_lst = df["Age Bias"].tolist()

age_bias_freqs = defaultdict(int)

for s in age_bias_lst:
    if isinstance(s, float):
        continue

    s = str(s).strip().lower()
    s = s.replace("_", " ")

    if s == "no":
        continue

    match = re.search(r"\((.*?)\)", s)
    if not match:
        continue

    items = match.group(1)
    for label in items.split(","):
        lab = label.strip()
        if lab:
            age_bias_freqs[lab] += 1

age_bias_freqs

In [ ]:
from pathlib import Path
import sys

_repo_root = Path.cwd()
if _repo_root.name == "demographic_bias":
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / "conclusions"))

from demographic_bucket_maps import (
    AGE_BIAS_EXTRA_RAW_TO_ALIAS,
    AGE_GROUPS,
    aggregate_bias_freqs,
    age_rows_sorted_by_mentions,
)

counts_by_bucket = aggregate_bias_freqs(
    dict(age_bias_freqs),
    AGE_GROUPS,
    extra_raw_to_alias=AGE_BIAS_EXTRA_RAW_TO_ALIAS,
)
groups_ordered, counts_ordered = age_rows_sorted_by_mentions(counts_by_bucket)
counts_by_bucket

In [ ]:
label_rotation = 0
x_tick_labelsize = 18
SHOW_AXIS_TITLES = True
axis_title_fontsize = 22
count_label_pad = 0.4
count_label_fontsize = 16
SAVE_SVG = True
svg_path = "age.svg"
ylabel = "Age group"

groups = groups_ordered
counts = counts_ordered
plot_df = pd.DataFrame({"group": groups, "count": counts})

n_cats = len(groups)
fig, ax = plt.subplots(figsize=(12, max(8, 0.38 * n_cats)))

sns.barplot(
    data=plot_df,
    y="group",
    x="count",
    ax=ax,
    color="royalblue",
    edgecolor="black",
    linewidth=1,
    order=plot_df["group"],
)
sns.despine(ax=ax)

xmax_plot = int(math.ceil(max(counts) / 5.0)) * 5 + 5 if counts else 10
x_step = 10 if xmax_plot >= 40 else 5
x_gap = count_label_pad

for i, count in enumerate(counts):
    ax.text(count + x_gap, i, f"{count}", ha="left", va="center", fontsize=count_label_fontsize)

ax.set_xlim(0, xmax_plot)
ax.set_xticks(range(0, xmax_plot + 1, x_step))

if SHOW_AXIS_TITLES:
    ax.set_xlabel("Frequency", fontsize=axis_title_fontsize)
    ax.set_ylabel(ylabel, fontsize=axis_title_fontsize, labelpad=4)

ax.tick_params(axis="x", labelsize=x_tick_labelsize)
ax.tick_params(axis="y", labelsize=x_tick_labelsize, pad=2)
ax.set_yticklabels(ax.get_yticklabels(), rotation=label_rotation, ha="right")

longest = max((len(str(t.get_text())) for t in ax.get_yticklabels()), default=10)
fig.subplots_adjust(
    left=min(0.38, max(0.11, 0.085 + 0.0058 * longest)),
    right=0.98,
    top=0.98,
    bottom=0.08,
)
if SAVE_SVG:
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
fig.savefig("age.png", dpi=200, bbox_inches="tight")
plt.show()